# Field Layout
Draws SAY-compliant markings for soccer field polygons in ArcGIS Online.

**Before running:** fill in the item IDs below. Leave `FIELD_IDS` empty to process all fields.

In [ ]:
# ── Cell 1: Configuration ────────────────────────────────────────────────
# Item ID of the input field polygons feature layer
FIELDS_ITEM_ID = ""  # TODO: set this

# Item IDs of the four output feature layers (created manually in ArcGIS Online)
LINES_ITEM_ID   = ""  # TODO: set this
CIRCLES_ITEM_ID = ""  # TODO: set this
MASKS_ITEM_ID   = ""  # TODO: set this
POINTS_ITEM_ID  = ""  # TODO: set this

# Optional: limit processing to specific field OBJECTIDs (empty list = all)
FIELD_IDS = []  # e.g. [1, 2, 5]

# Name of the Pitch Type attribute on the field polygon layer
PITCH_TYPE_FIELD = "Pitch Type"

print("Configuration loaded.")

In [ ]:
# ── Cell 2: Connect & Load ────────────────────────────────────────────────
import os
from arcgis.gis import GIS
from arcgis.features import FeatureLayer

# Auto-detect environment: ArcGIS Online notebook vs local Jupyter
try:
    gis = GIS("home")
    print("Connected via ArcGIS Online session.")
except Exception:
    gis = GIS("https://www.arcgis.com")
    print(f"Connected as {gis.properties.user.username}")

# Validate all item IDs are set
missing = [name for name, val in [
    ("FIELDS_ITEM_ID", FIELDS_ITEM_ID),
    ("LINES_ITEM_ID", LINES_ITEM_ID),
    ("CIRCLES_ITEM_ID", CIRCLES_ITEM_ID),
    ("MASKS_ITEM_ID", MASKS_ITEM_ID),
    ("POINTS_ITEM_ID", POINTS_ITEM_ID),
] if not val]
if missing:
    raise ValueError(f"Missing item IDs in Cell 1: {missing}")

# Load feature layers
fields_item   = gis.content.get(FIELDS_ITEM_ID)
lines_fl      = gis.content.get(LINES_ITEM_ID).layers[0]
circles_fl    = gis.content.get(CIRCLES_ITEM_ID).layers[0]
masks_fl      = gis.content.get(MASKS_ITEM_ID).layers[0]
points_fl     = gis.content.get(POINTS_ITEM_ID).layers[0]
fields_fl     = fields_item.layers[0]

# Read fill color per pitch type from source layer renderer
pitch_colors = {}  # e.g. {"7v7": [0, 128, 0, 255], ...}
renderer = fields_item.layers[0].properties.get("drawingInfo", {}).get("renderer", {})
if renderer.get("type") == "uniqueValue":
    for info in renderer.get("uniqueValueInfos", []):
        value = info["value"]
        color = info.get("symbol", {}).get("color", [0, 0, 0, 0])
        pitch_colors[value] = color
    print(f"Extracted colors for pitch types: {list(pitch_colors.keys())}")
else:
    print("Warning: source layer renderer is not UniqueValue. Masks will use transparent fill.")

print("Layers loaded.")

In [ ]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────
from geometry import yards_to_native
from field_markings import build_field_markings
from arcgis.geometry import Polyline, Polygon, Point
from arcgis.features import Feature
print("Imports OK.")